Intent de entrenament d'un model que predigui el millor possible un paràmetre

In [2]:
print("Hola")

Hola


In [3]:
import os
import sys
print(os.getcwd())

sys.path.append(r'c:\Users\jmass\Documents\GitHub\TFG_GEINF\src\data')
from DatasetGenerator import DatasetGenerator


c:\Users\jmass\Documents\GitHub\TFG_GEINF\notebooks\experiments


In [4]:
import pandas as pd

num_particules = 10000

dg = DatasetGenerator(num_particules)

In [5]:
data = dg.generate_BrownianMotion(random_params=True, num_passos=200)

In [7]:
data["attributes"][0]["delta_t"]

1.558825546182437

In [8]:
df = data[["coords", "attributes"]].copy()
df["delta_t"] = df["attributes"].apply(lambda x: x["delta_t"])
df = df[["coords", "delta_t"]]

In [9]:
df

,coords,delta_t
0,"[[0.0, 1.9888547599166528, 0.21261027373319208...",1.558826
1,"[[0.0, 0.5692195061484469, 1.4667550354999246,...",0.948925
2,"[[0.0, -5.446419334835012, -9.326188696822229,...",4.052302
3,"[[0.0, -3.0569864992181284, -0.673306577555023...",1.271402
4,"[[0.0, 0.8101673062803707, 1.0019615773901192,...",0.437264
...,...,...
9995,"[[0.0, -2.830194947250406, 4.6192222010545, 4....",2.965641
9996,"[[0.0, 1.4564358707323564, -2.7964194001874807...",1.999956
9997,"[[0.0, -0.5301250741543972, 3.037708093391624,...",4.942311
9998,"[[0.0, 0.9935729575275732, 1.3543856948678459,...",2.466858


In [10]:
print(df.columns)

Index(['coords', 'delta_t'], dtype='object')


In [11]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

In [12]:
# 1. Convertir las coordenadas a tensor 3D
# Asumiendo que 'data' es tu DataFrame
coords_list = [np.stack([row[0], row[1]], axis=1) for row in df['coords']]
X = np.array(coords_list)  # Forma (n_samples, seq_length, 2)
X_tensor = torch.tensor(X, dtype=torch.float32)

In [13]:
# 2. Convertir las etiquetas a tensor de valores reales
df["delta_t"] = df["delta_t"].apply(lambda x: (x - df["delta_t"].min()) / (df["delta_t"].max() - df["delta_t"].min()))
labels = df['delta_t'].values  # Suponiendo que 'sigma' contiene los valores reales
y_tensor = torch.tensor(labels, dtype=torch.float32)

In [14]:
df["delta_t"]

0       0.297645
1       0.173162
2       0.806576
3       0.238981
4       0.068729
          ...   
9995    0.584783
9996    0.387682
9997    0.988231
9998    0.482979
9999    0.652037
Name: delta_t, Length: 10000, dtype: float64

In [15]:
X_tensor.shape

torch.Size([10000, 200, 2])

In [16]:
# 3. Reorganizar dimensiones para Conv1D (n_samples, channels, seq_length)
X_tensor = X_tensor.permute(0, 2, 1)  # Nueva forma: (n, 2, m)

In [17]:
X_tensor.shape

torch.Size([10000, 2, 200])

In [18]:
# 4. Dividir datos en entrenamiento y prueba (ejemplo simple)
train_size = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:train_size], X_tensor[train_size:]
y_train, y_test = y_tensor[:train_size], y_tensor[train_size:]

In [19]:
# 5. Crear DataLoader
batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [18]:
# 6. Definir la arquitectura CNN
class TrajectoryCNN(nn.Module):
    def __init__(self, input_channels, seq_length):
        super().__init__()
        self.first_conv = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv_32_64 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv_64_128 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 4))
        self.fc1 = nn.Linear(4, 256)
        self.fc2 = nn.Linear(256, 1)

    
    def forward(self, data):
        a_resnet = self.first_conv(data)
        a = self.conv1_32(a_resnet)
        a = self.conv2_32(a)
        a = a + a_resnet
        a = self.max_pool(a)
        b_resnet = self.conv_32_64(a)
        b = self.conv1_64(b_resnet)
        b = self.conv2_64(b)
        b = b + b_resnet
        b = self.max_pool(b)
        c_resnet = self.conv_64_128(b)
        c = self.conv1_128(c_resnet)
        c = self.conv2_128(c)
        c = c + c_resnet
        c = self.avg_pool(c)
        c = c.view(c.size(0), -1)  # Aplanar la salida
        c = F.relu(self.fc1(c))
        c = self.fc2(c)
        c = torch.clamp(c, min=-1, max=1)
        return c

In [19]:
# 6. Definir la arquitectura CNN
class TrajectoryCNN(nn.Module):
    def __init__(self, input_channels, seq_length):
        super().__init__()
        self.first_conv = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv_32_64 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv_64_128 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool1d(4)
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )
        self.fc1 = nn.Linear(64, 256)
        self.fc2 = nn.Linear(256, 1)

    
    def forward(self, data):
        a_resnet = self.first_conv(data)
        a = self.conv1_32(a_resnet)
        a = self.conv2_32(a)
        a = a + a_resnet
        a = self.max_pool(a)
        b_resnet = self.conv_32_64(a)
        b = self.conv1_64(b_resnet)
        b = self.conv2_64(b)
        b = b + b_resnet
        b = self.max_pool(b)
        c_resnet = self.conv_64_128(b)
        c = self.conv1_128(c_resnet)
        c = self.conv2_128(c)
        c = c + c_resnet
        c = self.avg_pool(c)
        c = c.permute(0, 2, 1)
        lstm_out, _ = self.lstm(c)
        lstm_last = lstm_out[:, -1, :]
        c = F.relu(self.fc1(lstm_last))
        c = self.fc2(c)
        c = torch.clamp(c, min=-1, max=1)
        return c

In [20]:
import torch.nn as nn

class TrajectoryCNN(nn.Module):
    def __init__(self, input_channels, seq_length):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(input_channels, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),  # [batch, 128, 1]
        )
        self.head = nn.Sequential(
            nn.Flatten(),             # [batch, 128]
            nn.Linear(128, 64)  ,
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Tanh()                 # Output ∈ [-1, 1]
        )

    def forward(self, x):
        x = self.encoder(x)
        return self.head(x)

In [21]:
# 7. Inicializar el modelo
model = TrajectoryCNN(
    input_channels=2,
    seq_length=X_tensor.shape[2]
)


In [22]:
model.first_conv.weight.shape

AttributeError: 'TrajectoryCNN' object has no attribute 'first_conv'

In [23]:
model.fc2.weight.shape

AttributeError: 'TrajectoryCNN' object has no attribute 'fc2'

In [22]:
# 8. Configurar entrenamiento
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [23]:
# 9. Entrenar el modelo
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, _ in train_loader:  # Ignorar las etiquetas
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.squeeze(), y_train[:len(inputs)])  # Comparar con valores reales
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")


Epoch [1/10], Loss: 0.1010
Epoch [2/10], Loss: 0.0873
Epoch [3/10], Loss: 0.0871
Epoch [4/10], Loss: 0.0871
Epoch [5/10], Loss: 0.0870
Epoch [6/10], Loss: 0.0870
Epoch [7/10], Loss: 0.0870
Epoch [8/10], Loss: 0.0870
Epoch [9/10], Loss: 0.0870
Epoch [10/10], Loss: 0.0871


In [24]:
# 10. Evaluación (opcional)
with torch.no_grad():
    test_outputs = model(X_test).squeeze()  # Asegurarse de que las dimensiones coincidan
    mse = criterion(test_outputs, y_test)  # Calcular el MSE
    print(f'Test MSE: {mse.item():.4f}')


Test MSE: 0.0910


In [26]:
# Seleccionar un ejemplo puntual de X_test
example_input = X_test[0].unsqueeze(0)  # Añadir dimensión batch

# Realizar la predicción
model.eval()  # Asegurarse de que el modelo esté en modo evaluación
with torch.no_grad():
    prediction = model(example_input).item()  # Obtener el valor predicho como un escalar

print(f"Predicción para el ejemplo puntual: {prediction}")
print(f"Valor real: {y_test[0].item()}")  # Mostrar el valor real para comparación

Predicción para el ejemplo puntual: 0.40041396021842957
Valor real: 0.7468441724777222


In [27]:
# Guardar solo los parámetros del modelo
torch.save(model.state_dict(), 'model_BrownianMotion_delta_t_prediction.pth')

# Si deseas guardar el modelo completo
torch.save(model, 'model_BrownianMotion_delta_t_prediction_complete.pth')

In [ ]:
# Cargar solo los parámetros del modelo
model = TrajectoryCNN(
    input_channels=2,
    seq_length=X_tensor.shape[2],  # m (longitud de la secuencia)
    num_classes=len(le.classes_)
)
model.load_state_dict(torch.load('trajectory_cnn_model_2.pth'))
model.eval()

# Si deseas cargar el modelo completo
# model = torch.load('trajectory_cnn_model_complete.pth')
# model.eval()